In [1]:
import os, sys
from pathlib import Path

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "app").is_dir() and (d / "requirements.txt").exists():
            return d
    raise RuntimeError(f"Couldn't find the project root above {here}")

ROOT = find_project_root()
os.chdir(ROOT)                 # so paths like "sample_repos/aoi" resolve from the top
sys.path.insert(0, str(ROOT))  # so "from app..." imports work

from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

print("Project root:", ROOT)

Project root: /Users/adamolderr/Desktop/workspace/ao-borg


In [2]:
from app.indexing.chunker import iter_python_files, chunk_file
import tree_sitter_python as tspython

root = Path("sample_repos/aoi").resolve()
chunks = [d for f in iter_python_files(root) for d in chunk_file(f, root)]
print("total chunks:", len(chunks))
for d in chunks[:10]:
    print(d.metadata["path"], d.metadata["symbol"], d.metadata["start_line"], d.metadata["end_line"])

total chunks: 35
blob_upload.py <module> 1 40
scratch_pad.py _to_dict 19 22
scratch_pad.py _employee_to_text 25 38
scratch_pad.py _task_to_text 40 54
scratch_pad.py _proficiency_multiplier 56 83
scratch_pad.py mapEmployeesToTasks 85 204
scratch_pad.py TaskEmployeeCandidate 287 302
scratch_pad.py <module> 1 302
test_mcp.py main 7 53
test_mcp.py <module> 1 56


In [3]:
print(chunks[5].page_content)

def mapEmployeesToTasks(
    embedding_client,
    embedding_model_name,
    employees,
    tasks,
    mode="productive",
):
    """
    Assign employees to unique tasks using numeric embeddings.

    Each employee and task is embedded as a numeric vector.
    Cosine similarity measures semantic relevance.
    Proficiency multipliers adjust scores by mode.
    The Hungarian algorithm finds the best overall assignment.

    Returns a dictionary keyed by task ID, or task index if no ID exists.
    """

    if mode not in PROFICIENCY_MULTIPLIERS:
        raise ValueError(
            "mode must be 'productive' or 'knowledge_transfer'"
        )

    if not employees or not tasks:
        return {}

    employee_texts = [
        _employee_to_text(employee)
        for employee in employees
    ]
    task_texts = [
        _task_to_text(task)
        for task in tasks
    ]

    # Generate numeric embeddings using your resource-level client.
    employee_response = embedding_client.embeddi

In [4]:
from azure.identity import (DefaultAzureCredential, get_bearer_token_provider)

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

In [5]:
import os
from urllib.parse import urlparse
from openai import OpenAI

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

base_url = f"https://{urlparse(PROJECT_ENDPOINT).hostname}/openai/v1/"
print("base_url:", base_url)
print("model:", repr(os.environ["FOUNDRY_EMBEDDING_MODEL_NAME"]))

client = OpenAI(base_url=base_url, api_key=token_provider)
r = client.embeddings.create(model=os.environ["FOUNDRY_EMBEDDING_MODEL_NAME"], input="hello")
print(len(r.data[0].embedding))

base_url: https://foundry-ao-borg-0.services.ai.azure.com/openai/v1/
model: 'text-embedding-3-large'
3072


In [6]:
from langchain_openai import OpenAIEmbeddings

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
EMBEDDING_MODEL_NAME = os.environ["FOUNDRY_EMBEDDING_MODEL_NAME"]

print('PROJECT_ENDPOINT', PROJECT_ENDPOINT)
print('EMBEDDING_MODEL_NAME ', EMBEDDING_MODEL_NAME)

def get_embeddings(token_provider):
    host = urlparse(PROJECT_ENDPOINT).hostname   # foundry-ao-borg-0.services.ai.azure.com
    return OpenAIEmbeddings(
        model=os.environ["FOUNDRY_EMBEDDING_MODEL_NAME"],
        base_url=f"https://{host}/openai/v1/",
        api_key=token_provider,
        check_embedding_ctx_length=False,
    )

PROJECT_ENDPOINT https://foundry-ao-borg-0.services.ai.azure.com/api/projects/foundry-ao-borg-0-proj
EMBEDDING_MODEL_NAME  text-embedding-3-large


In [7]:
vec = get_embeddings(token_provider).embed_query("hello")
print(len(vec))

3072


In [8]:
from app.indexing.store import _store, index_repo, get_retriever

In [9]:
s = _store(token_provider)
print(type(s))
print("chunks in store:", len(s.get()["ids"]))

<class 'langchain_chroma.vectorstores.Chroma'>
chunks in store: 0


In [10]:
import openai
from app.indexing.store import _store

emb = _store(token_provider)._embedding_function
print("base:", emb.openai_api_base)
print("model:", repr(emb.model))

try:
    emb.embed_query("hello")
except openai.NotFoundError as e:
    print("URL:", e.response.url)
    print("BODY:", repr(e.response.text))

base: https://foundry-ao-borg-0.services.ai.azure.com/openai/v1/
model: 'text-embedding-3-large'


In [11]:
emb = _store(token_provider)._embedding_function
print("base:", emb.openai_api_base)
print(len(emb.embed_query("hello")))   # expect 3072

base: https://foundry-ao-borg-0.services.ai.azure.com/openai/v1/
3072


In [12]:
emb = _store(token_provider)._embedding_function
print("base:", emb.openai_api_base)
print("check ctx length:", emb.check_embedding_ctx_length)

base: https://foundry-ao-borg-0.services.ai.azure.com/openai/v1/
check ctx length: False


In [13]:
import os, openai
from urllib.parse import urlparse
from openai import OpenAI

host = urlparse(PROJECT_ENDPOINT).hostname
resource = host.split(".")[0]
name = os.environ["FOUNDRY_EMBEDDING_MODEL_NAME"]
print("deployment name used:", repr(name))

for base in [f"https://{host}/openai/v1/",
             f"https://{resource}.openai.azure.com/openai/v1/"]:
    client = OpenAI(base_url=base, api_key=token_provider)
    try:
        r = client.embeddings.create(model=name, input="hello")
        print("OK  ", base, "->", len(r.data[0].embedding))
    except openai.APIStatusError as e:
        print("FAIL", base, e.status_code, repr(e.response.text))

deployment name used: 'text-embedding-3-large'
OK   https://foundry-ao-borg-0.services.ai.azure.com/openai/v1/ -> 3072
OK   https://foundry-ao-borg-0.openai.azure.com/openai/v1/ -> 3072


In [14]:
n = index_repo("sample_repos/aoi", token_provider)
print("chunks indexed:", n)
print("chunks in store:", len(_store(token_provider).get()["ids"]))

root /Users/adamolderr/Desktop/workspace/ao-borg/sample_repos/aoi
chunks indexed: 35
chunks in store: 35
